# 09 · Joint momentum, volume and overnight RET ensemble

Combine the original 15 momentum/state inputs, eight network-paper individual
inputs, six relative-volume inputs and current overnight gap. Remove exact
feature duplicates at the model's float32 precision; retain different
normalizations. No correlation or outcome-based feature selection.

The forecast rows, targets, labels, seeds, tree settings and N=2/3 weights are
unchanged. All results remain development research with optimistic same-open
information/fills and the existing source/eligibility limitations.

In [ ]:
from pathlib import Path
import json,os,sys
PROJECT_DIR=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"scripts/joint_features.py").exists())
sys.path.insert(0,str(PROJECT_DIR/"scripts"))
RUN_DIR=PROJECT_DIR/"data/models/09_xgb_ret_joint100_v1"
BASELINE_DIR=PROJECT_DIR/"data/models/07_xgb_ret_volume_overnight100_v1"
FEATURE_DIR=PROJECT_DIR/"data/features/09_joint_momentum_volume_gap_v1"
REPORT_DIR=PROJECT_DIR/"data/reports/09_train_xgb_ret_joint"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR",str(REPORT_DIR/".matplotlib"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display,Image,Markdown
from ret_backtest import performance
from joint_features import MODEL_FEATURES
RUN_TRAINING=False  # This notebook inspects only; the gated CLI starts training.
manifest=json.loads((FEATURE_DIR/"MANIFEST.json").read_text())
assert manifest["model_features"]==MODEL_FEATURES and len(set(MODEL_FEATURES))==30
assert manifest["summary"]["cohort_labels_outcomes_identical"]
display(pd.DataFrame.from_dict(manifest["summary"],orient="index",columns=["feature_release"]))
display(pd.DataFrame({"feature":MODEL_FEATURES}))

## Schedule and controls

Training cannot start before **2026-11-06 15:40 Europe/Vienna (14:40 UTC)**,
including its runtime benchmark. Forty outer quarters, 100 members each, twelve
training months and three forecast months; initial training 2016. Ten CPU
workers, one thread each. No boosting-round maximum, fixed RAM ceiling,
available-memory launch threshold, runtime cutoff or fit-time budget. Stop only
on thirty non-improving validation rounds, completion, failure or explicit user
interruption. The start gate is a lower bound, not a deadline.

In [ ]:
status=json.loads((RUN_DIR/"RUN_STATUS.json").read_text()) if (RUN_DIR/"RUN_STATUS.json").exists() else {"status":"prepared"}
display(pd.DataFrame.from_dict(status,orient="index",columns=["run_status"]))
COMPLETE=(RUN_DIR/"RESULTS.json").exists() and status.get("status")=="complete"
if not COMPLETE: display(Markdown("Training is scheduled/running. Full-ensemble performance is withheld until all 4,000 models finish."))

## Controlled comparison against the completed fifteen-input model

Compare both feature sets and N values on the same observed dates. Keep cash
sessions. Unknown selected outcomes remain unknown; conditional curves do not
certify a complete executable path. The original momentum block is the only
statistical addition. The current overnight-gap information assumption remains
unchanged.

In [ ]:
if COMPLETE:
    books=[]
    for label,folder in [("combined15",BASELINE_DIR),("joint30",RUN_DIR)]:
        b=pd.read_parquet(folder/"portfolio_daily.parquet");b["feature_set"]=label;books.append(b)
    books=pd.concat(books,ignore_index=True)
    wide=books.pivot(index="trade_date",columns=["feature_set","n_per_side"],values="gross_return")
    common=wide.notna().all(axis=1);dates=common[common].index
    records=[]
    for (label,n),book in books.groupby(["feature_set","n_per_side"]):
        observed=book.set_index("trade_date").loc[dates]
        for cost in [0,1,5,10]:
            records.append(dict(feature_set=label,n_per_side=n,cost_bps_per_side=cost,
                calendar_sessions=len(book),omitted_dates=int((~common).sum()),
                metric_scope="four_way_common_observed_dates_conditional",
                **performance(observed.gross_return-observed.trade_notional*cost/10000)))
    comparison=pd.DataFrame(records);comparison.to_parquet(REPORT_DIR/"common_date_comparison.parquet",index=False)
    common.to_frame("included").to_parquet(REPORT_DIR/"common_date_mask.parquet")
    display(comparison)
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,cost in zip(axes,[0,1]):
        for (label,n),book in books.groupby(["feature_set","n_per_side"]):
            b=book.set_index("trade_date").loc[dates]
            ax.plot(dates,(1+b.gross_return-b.trade_notional*cost/10000).cumprod(),label=f"{label}, N={n}")
        ax.set_title(f"Conditional equity, {cost} bp/side");ax.legend(fontsize=8);ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(REPORT_DIR/"joint_comparison.png",dpi=150);plt.show()

In [ ]:
if COMPLETE:
    fingerprints=["split_seed","booster_seed","split_train_index_sha256","split_validation_index_sha256","prediction_row_key_sha256"]
    members=0
    for folder in sorted((RUN_DIR/"folds").iterdir()):
        if not folder.is_dir():continue
        current=json.loads((folder/"FOLD.json").read_text())
        previous=json.loads((BASELINE_DIR/"folds"/folder.name/"FOLD.json").read_text())
        np.testing.assert_allclose(current["payoff_vector_class_1_to_10"],previous["payoff_vector_class_1_to_10"],atol=1e-12,rtol=0)
        paths=list((folder/"members").glob("member_*.json"));assert len(paths)==100
        for path in paths:
            a=json.loads(path.read_text());b=json.loads((BASELINE_DIR/"folds"/folder.name/"members"/path.name).read_text())
            assert all(a[k]==b[k] for k in fingerprints)
        members+=len(paths)
    assert members==4000
    display(pd.DataFrame({label:json.loads((folder/"RESULTS.json").read_text())["classification"] for label,folder in [("combined15",BASELINE_DIR),("joint30",RUN_DIR)]}))
    (REPORT_DIR/"COMPARISON_AUDIT.json").write_text(json.dumps(dict(members=members,split_row_fingerprints_match=True,payoffs_match=True,cohort_labels_outcomes_unchanged=True),indent=2))

## Importance, explained variance and calibration

Gain shares describe in-training classifier splits, not explained return
variance or causal effects. EV and R² use saved ensemble RET forecasts on exact
matched observed stock-days. Negative scores remain visible. Different
normalizations can share correlated information despite having no identical
columns. No fresh final holdout or delayed-execution evidence is claimed.

In [ ]:
if COMPLETE:
    from analyze_volume_overnight_results import analyze
    diagnostics=analyze(PROJECT_DIR,experiment="joint30")
    display(pd.read_parquet(diagnostics/"feature_importance.parquet"))
    display(pd.read_parquet(diagnostics/"explained_variance.parquet"))
    for filename in ["feature_importance.png","feature_importance_by_quarter.png","out_of_sample_explained_variance.png","explained_variance_by_year.png","out_of_sample_return_calibration.png"]:
        display(Image(filename=str(diagnostics/filename)))
    (REPORT_DIR/"SUMMARY.json").write_text(json.dumps(dict(status="complete",models=4000,common_observed_dates=len(dates),omitted_dates=int((~common).sum()),comparison=comparison.to_dict("records")),indent=2,default=str))